In [1]:
!pip install transformers datasets evaluate jiwer peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.7 MB/s eta 0:00:00


In [2]:
!pip install --upgrade torchao
!pip install transformers datasets evaluate jiwer peft accelerate librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
import os
import librosa
import torch
import pandas as pd
from collections import Counter
from transformers import pipeline
import evaluate

wer_metric = evaluate.load("wer")

# 1. Setup paths for your long audio file
audio_file_path = "/content/Full Narration_MarauliKhurad.m4a"
audio_array, sampling_rate = librosa.load(audio_file_path, sr=16000)

models_to_test = [
     "Garden2006/whisper-large-v3-turbo-gurmukhi-lora",
     "abhi8799/whisper-large-v3-turbo-gurmukhi-lora",
     "abhi8799/whisper-small-gurmukhi-lora",
     "KaliNangia/whisper-large-v3-turbo-gurmukhi-lora",
     "KaliNangia/whisper-medium-gurmukhi-lora"
]

raw_transcripts = {}

# Step A: Generate and save transcripts for all models
for model_id in models_to_test:
    print(f"Running inference for {model_id}...")
    try:
        asr_pipeline = pipeline(
            "automatic-speech-recognition",
            model=model_id,
            device=0 if torch.cuda.is_available() else -1,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )

        prediction = asr_pipeline(
            {"array": audio_array, "sampling_rate": sampling_rate},
            chunk_length_s=30,
            return_timestamps=True,
            generate_kwargs={"language": "punjabi", "task": "transcribe"}
        )["text"].strip()

        raw_transcripts[model_id] = prediction

        # Format filename per Abhishek's instructions: <HF_USERNAME>_<MODEL>.txt
        hf_username = model_id.split("/")[0]
        model_short = "turbo" if "turbo" in model_id else ("medium" if "medium" in model_id else ("small" if "small" in model_id else "large"))

        with open(f"{hf_username}_{model_short}.txt", "w", encoding="utf-8") as f:
            f.write(prediction)

    except Exception as e:
        print(f"❌ Failed to run {model_id}: {e}")

# Step B: Construct a "Pseudo-Ground Truth" via Majority Voting
# Split transcripts into lists of words
model_word_lists = {m: t.split() for m, t in raw_transcripts.items()}
max_words = max(len(w_list) for w_list in model_word_lists.values())

pseudo_ground_truth_words = []
for i in range(max_words):
    current_pool = []
    for m in models_to_test:
        if i < len(model_word_lists[m]):
            current_pool.append(model_word_lists[m][i])
    if current_pool:
        # Find the most common word at this position across the models
        most_common_word = Counter(current_pool).most_common(1)[0][0]
        pseudo_ground_truth_words.append(most_common_word)

pseudo_ground_truth_text = " ".join(pseudo_ground_truth_words)

# Step C: Benchmarking against the Consensus Profile
consensus_results = []
for model_id in models_to_test:
    if model_id in raw_transcripts:
        # Measure how much the model deviates from the group agreement baseline
        cwer_score = wer_metric.compute(
            predictions=[raw_transcripts[model_id]],
            references=[pseudo_ground_truth_text]
        )
        consensus_results.append({
            "Model Name": model_id,
            "Consensus WER (cWER)": round(cwer_score, 4),
            "Status": "Evaluated via Cross-Model Voting"
        })

# Compile into data frame
df_report = pd.DataFrame(consensus_results)
print("\n--- Benchmarking Complete ---")
print(df_report)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/tmp/ipykernel_2537/3931164880.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(audio_file_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Running inference for Garden2006/whisper-large-v3-turbo-gurmukhi-lora...


adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.



--- Benchmarking Complete ---
                                        Model Name  Consensus WER (cWER)  \
0  Garden2006/whisper-large-v3-turbo-gurmukhi-lora                0.0688   
1    abhi8799/whisper-large-v3-turbo-gurmukhi-lora                0.5888   
2             abhi8799/whisper-small-gurmukhi-lora                0.8659   
3  KaliNangia/whisper-large-v3-turbo-gurmukhi-lora                0.6757   
4          KaliNangia/whisper-medium-gurmukhi-lora                0.7346   

                             Status  
0  Evaluated via Cross-Model Voting  
1  Evaluated via Cross-Model Voting  
2  Evaluated via Cross-Model Voting  
3  Evaluated via Cross-Model Voting  
4  Evaluated via Cross-Model Voting  


In [2]:
# Create the markdown text structure
report_md = f"""# Benchmark Report: Reference-Free Transcription Evaluation

## 1. Objective & Core Research Problem
When evaluating Automatic Speech Recognition (ASR) systems on production domains or unique localized telemetry datasets, human-verified ground-truth transcripts are often unavailable. Standard Word Error Rate (WER) metrics cannot execute without a verified reference text baseline.

This report presents a reference-free alternative benchmark strategy by establishing a **Cross-Model Consensus Framework** utilizing 5 discrete team-fine-tuned Whisper model configurations.

## 2. Benchmark Methodology
Instead of validating accuracy against an absolute static human script, transcription reliability is measured via **Majority Voting (Token-Level Consensus)**:
1. **Inference Execution:** The same target file (`Full Narration_MarauliKhurad.m4a`) is processed independently by all 5 LoRA architectures using identical decoding parameters ($16\\text{{ kHz}}$ chunked stream processing, Punjabi localization).
2. **Pseudo-Ground Truth Construction:** A structural consensus vector is synthesized dynamically. For every sequential word slot ($i$), a token frequency matrix is evaluated. The word agreed upon by the majority of models is injected into the baseline reference string.
3. **Consensus WER (cWER) Evaluation:** Each individual model's original prediction stream is mapped against this artificial baseline using standard Levenshtein distance calculations. A lower **cWER** implies a model aligns closely with collective consensus, indicating higher operational stability and resistance to hallucination variations.

## 3. Benchmarking Matrix Results
The table below represents how closely each fine-tuned model variants adhere to the collective linguistic consensus:

{df_report.to_markdown(index=False)}

## 4. Key Engineering Insights
* **Consensus Drivers:** Models with the lowest cWER scores demonstrate high vocabulary alignment with alternative parameter pools, marking them as stable candidate architectures for zero-shot tasks where a ground truth is missing.
* **Outlier Variance:** Higher cWER metrics flag architectures that display structural variations, alternative word order selections, or localized phrase interpretations.

---
*Report automatically compiled and generated in Google Colab.*
"""

# Write out the final file
report_file_path = "Reference_Free_Benchmark_Report.md"
with open(report_file_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(f"📝 Final report successfully saved to {report_file_path}!")

📝 Final report successfully saved to Reference_Free_Benchmark_Report.md!


In [3]:
# --- ADD THESE LINES AT THE VERY END OF YOUR CODE CELL ---

# 1. Save the generated pseudo-transcript to a physical file
with open("consensus_pseudo_ground_truth.txt", "w", encoding="utf-8") as f_pseudo:
    f_pseudo.write(pseudo_ground_truth_text)
print("💾 Successfully saved consensus master file to: consensus_pseudo_ground_truth.txt")

# 2. Print a quick sneak peek of the text to verify it's there
print("\n👀 Pseudo-Transcript Snippet (First 100 characters):")
print(pseudo_ground_truth_text[:100] + "...")

💾 Successfully saved consensus master file to: consensus_pseudo_ground_truth.txt

👀 Pseudo-Transcript Snippet (First 100 characters):
ਸਾਸਿਕਾਲ ਜੀ ਮੀਟਿਂ ਸ਼ਜਿੂਲ ਮੜੋਲੀ ਖੁਰਦ ਦੇਟ ਦੋ ਅਪਰਾਇਲ ਦੋ ਦੋਹਿਆਰ ਛੱਬੀ ਬਿਲਜ ਮਡੋਲੀ ਖੁਰਦ ਮਡੋਲੀ ਖੁਰਦ ਬੁਲੋਕ ਬੁਲ...


In [ ]:
#!pip install google-genai

In [ ]:
# import os
# from google.colab import userdata
# from google.genai import client

# # 1. Initialize the modern GenAI Client using GOOGLE_API_KEY
# try:
#     # Safely passes your GOOGLE_API_KEY from Colab secrets to the environment variable
#     os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')
#     ai = client.Client()
#     print("✅ GenAI Client initialized successfully using GOOGLE_API_KEY!")
# except Exception as e:
#     print(f"❌ Authentication Failed: {e}")
#     print("👉 Make sure your secret key in the Colab sidebar (🔑) is named exactly 'GOOGLE_API_KEY' and Notebook access is toggled ON.")

# # 2. Structure ALL your complete transcripts
# transcripts_block = ""
# for model_name, text in raw_transcripts.items():
#     transcripts_block += f"### Model: {model_name}\nComplete Transcript:\n{text}\n\n"

# # 3. Craft the evaluation prompt
# judge_prompt = f"""
# You are an expert linguistic judge specializing in Punjabi (Gurmukhi script) Automatic Speech Recognition (ASR).
# Your task is to evaluate the outputs of 5 different ASR models that transcribed the same audio file.
# We do not have a human-verified ground-truth transcript, so you must judge them comparatively based on:
# 1. Linguistic Fluency & Punjabi grammar.
# 2. Structural stability (absence of typical ASR loops, repetition bugs, or character spam).
# 3. Contextual cohesion.

# Here are the transcripts:
# {transcripts_block}

# Please evaluate each model and output your analysis strictly in the following Markdown table format followed by a short summary of the clear winner:
# | Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
# | --- | --- | --- | --- |
# """

# # 4. Generate the judgment via the modern cloud SDK architecture
# # Only runs if 'ai' client was successfully created above
# if 'ai' in locals():
#     print("🤖 Cloud LLM Judge is processing the complete transcripts...")
#     response = ai.models.generate_content(
#         model='gemini-2.5-flash',
#         contents=judge_prompt,
#     )
#     judge_report = response.text

#     print("\n--- ⚖️ LLM Judge Report ---")
#     print(judge_report)

#     # 5. Save the generated report to a physical file for download
#     output_file_path = "LLM_Judge_ASR_Evaluation.md"
#     with open(output_file_path, "w", encoding="utf-8") as f:
#         f.write(judge_report)

#     print(f"\n📝 Successfully saved the evaluation report to: {output_file_path}")
#     print("You can download it directly from the Files sidebar pane on the left!")
# else:
#     print("\n❌ Execution halted: Client initialization failed. Check your API token configuration.")

In [9]:
import gc
import torch

# Clear variables from memory if they exist
for var in ['asr_pipeline', 'judge_pipeline', 'outputs']:
    if var in locals() or var in globals():
        exec(f"del {var}")

# Force garbage collection and flush CUDA cache
gc.collect()
torch.cuda.empty_cache()

print("🧹 GPU Memory cleared! You can now try running your Gemma cell again.")

🧹 GPU Memory cleared! You can now try running your Gemma cell again.


In [4]:
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00


In [ ]:
# # import os
# # import torch
# # from google.colab import userdata
# # from transformers import pipeline

# # # 1. Authenticate using your exact 'HF_token' secret name
# # try:
# #     os.environ["HF_TOKEN"] = userdata.get('HF_token')
# #     print("✅ Hugging Face token loaded successfully!")
# # except Exception as e:
# #     print(f"❌ Failed to load secret: {e}")

# # # 2. Initialize Gemma 2B Instruct locally on your T4 GPU
# # print("⏳ Loading local Gemma Judge model into GPU memory...")
# # judge_pipeline = pipeline(
# #     "text-generation",
# #     model="google/gemma-2-2b-it",
# #     device=0,                      # target the T4 GPU
# #     torch_dtype=torch.bfloat16     # optimizes VRAM usage to fit the T4
# # )

# # # 3. Structure the complete transcripts
# # transcripts_block = ""
# # for model_name, text in raw_transcripts.items():
# #     transcripts_block += f"### Model: {model_name}\nComplete Transcript:\n{text}\n\n"

# # # 4. Craft the evaluation prompt
# # judge_prompt = f"""You are an expert linguistic judge specializing in Punjabi (Gurmukhi script) Automatic Speech Recognition (ASR).
# # Your task is to evaluate the outputs of 5 different ASR models that transcribed the same audio file.
# # We do not have a human-verified ground-truth transcript, so you must judge them comparatively based on:
# # 1. Linguistic Fluency & Punjabi grammar.
# # 2. Structural stability (absence of typical ASR loops, repetition bugs, or character spam).
# # 3. Contextual cohesion.

# # Here are the transcripts:
# # {transcripts_block}

# # Please evaluate each model and output your analysis strictly in the following Markdown table format followed by a short summary of the clear winner:
# # | Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
# # | --- | --- | --- | --- |
# # """

# # # 5. Format message block for Gemma chat template architecture
# # messages = [{"role": "user", "content": judge_prompt}]

# # print("🤖 Local Gemma Judge is processing the complete transcripts...")
# # outputs = judge_pipeline(messages, max_new_tokens=1500)
# # judge_report = outputs[0]["generated_text"][-1]["content"]

# # print("\n--- ⚖️ LLM Judge Report ---")
# # print(judge_report)

# # # 6. Save the generated report to a physical file for download
# # output_file_path = "LLM_Judge_ASR_Evaluation.md"
# # with open(output_file_path, "w", encoding="utf-8") as f:
# #     f.write(judge_report)

# # print(f"\n📝 Successfully saved the evaluation report to: {output_file_path}")
# # print("You can download it directly from the Files sidebar pane on the left!")

# import os
# import torch
# from google.colab import userdata
# from transformers import pipeline, BitsAndBytesConfig

# # 1. Authenticate using your 'HF_token'
# try:
#     os.environ["HF_TOKEN"] = userdata.get('HF_token')
#     print("✅ Hugging Face token loaded successfully!")
# except Exception as e:
#     print(f"❌ Failed to load secret: {e}")

# # 2. Configure 4-bit quantization to drastically reduce memory usage
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# print("⏳ Loading local Gemma Judge model in 4-bit (Memory-Optimized)...")
# judge_pipeline = pipeline(
#     "text-generation",
#     model="google/gemma-2-2b-it",
#     model_kwargs={"quantization_config": quantization_config} # Saves ~8GB of VRAM!
# )

# # 3. Structure the complete transcripts
# transcripts_block = ""
# for model_name, text in raw_transcripts.items():
#     transcripts_block += f"### Model: {model_name}\nComplete Transcript:\n{text}\n\n"

# # 4. Craft the evaluation prompt
# judge_prompt = f"""You are an expert linguistic judge specializing in Punjabi (Gurmukhi script) Automatic Speech Recognition (ASR).
# Your task is to evaluate the outputs of the different ASR models that transcribed the same audio file.
# We do not have a human-verified ground-truth transcript, so you must judge them comparatively based on:
# 1. Linguistic Fluency & Punjabi grammar.
# 2. Structural stability (absence of typical ASR loops, repetition bugs, or character spam).
# 3. Contextual cohesion.

# Here are the transcripts:
# {transcripts_block}

# Please evaluate each model and output your analysis strictly in the following Markdown table format followed by a short summary of the clear winner:
# | Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
# | --- | --- | --- | --- |
# """

# # 5. Format message block for Gemma
# messages = [{"role": "user", "content": judge_prompt}]

# print("🤖 Local Gemma Judge is processing the complete transcripts...")
# with torch.no_grad():
#     outputs = judge_pipeline(messages, max_new_tokens=1500)

# judge_report = outputs[0]["generated_text"][-1]["content"]

# print("\n--- ⚖️ LLM Judge Report ---")
# print(judge_report)

# # 6. Save the generated report to a physical file for download
# output_file_path = "LLM_Judge_ASR_Evaluation.md"
# with open(output_file_path, "w", encoding="utf-8") as f:
#     f.write(judge_report)

# print(f"\n📝 Successfully saved the evaluation report to: {output_file_path}")

✅ Hugging Face token loaded successfully!
⏳ Loading local Gemma Judge model in 4-bit (Memory-Optimized)...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

NameError: name 'raw_transcripts' is not defined

In [ ]:
# The most failed output
# import os
# import torch
# from google.colab import userdata
# from transformers import pipeline, BitsAndBytesConfig

# # 1. Authenticate using your 'HF_token'
# try:
#     os.environ["HF_TOKEN"] = userdata.get('HF_token')
#     print("✅ Hugging Face token loaded successfully!")
# except Exception as e:
#     print(f"❌ Failed to load secret: {e}")

# # -----------------------------------------------------------------
# # 📂 READ THE THREE TRANSCRIPT FILES DIRECTLY FROM YOUR SIDEBAR
# # -----------------------------------------------------------------
# files_to_check = {
#     "Garden2006/whisper-large-v3-turbo-gurmukhi-lora": "Garden2006_turbo.txt",
#     "abhi8799/whisper-large-v3-turbo-gurmukhi-lora": "abhi8799_turbo.txt",
#     "abhi8799/whisper-small-gurmukhi-lora": "abhi8799_small.txt"
# }

# transcripts_block = ""
# print("\n📂 Reading the three saved transcript files...")

# for model_name, filename in files_to_check.items():
#     if os.path.exists(filename):
#         with open(filename, "r", encoding="utf-8") as f:
#             content = f.read().strip()
#             # Append this file's text directly to the judge's prompt block
#             transcripts_block += f"### Model: {model_name}\nComplete Transcript:\n{content}\n\n"
#             print(f"  ✔️ Successfully read: {filename}")
#     else:
#         print(f"  ❌ Missing file: {filename} (Please verify it exists in your left sidebar)")

# # 2. Configure 4-bit quantization to save your T4 GPU memory
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# print("\n⏳ Loading local Gemma Judge model in 4-bit (Memory-Optimized)...")
# judge_pipeline = pipeline(
#     "text-generation",
#     model="google/gemma-2-2b-it",
#     model_kwargs={"quantization_config": quantization_config}
# )

# # 3. Craft the final evaluation prompt using the structured file data
# judge_prompt = f"""You are an expert linguistic judge specializing in Punjabi (Gurmukhi script) Automatic Speech Recognition (ASR).
# Your task is to evaluate the outputs of the different ASR models that transcribed the same audio file.
# We do not have a human-verified ground-truth transcript, so you must judge them comparatively based on:
# 1. Linguistic Fluency & Punjabi grammar.
# 2. Structural stability (absence of typical ASR loops, repetition bugs, or character spam).
# 3. Contextual cohesion.

# Here are the transcripts:
# {transcripts_block}

# Please evaluate each model and output your analysis strictly in the following Markdown table format followed by a short summary of the clear winner:
# | Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
# | --- | --- | --- | --- |
# """

# # 4. Format message block for Gemma
# messages = [{"role": "user", "content": judge_prompt}]

# print("\n🤖 Local Gemma Judge is processing the transcripts from your files...")
# with torch.no_grad():
#     outputs = judge_pipeline(messages, max_new_tokens=1500)

# judge_report = outputs[0]["generated_text"][-1]["content"]

# print("\n--- ⚖️ LLM Judge Report ---")
# print(judge_report)

# # 5. Save the generated report to a physical file for download
# output_file_path = "LLM_Judge_ASR_Evaluation.md"
# with open(output_file_path, "w", encoding="utf-8") as f:
#     f.write(judge_report)

# print(f"\n📝 Successfully saved the evaluation report to: {output_file_path}")
# print("You can download it directly from your Files sidebar pane on the left!")

✅ Hugging Face token loaded successfully!

📂 Reading the three saved transcript files...
  ✔️ Successfully read: Garden2006_turbo.txt
  ✔️ Successfully read: abhi8799_turbo.txt
  ✔️ Successfully read: abhi8799_small.txt

⏳ Loading local Gemma Judge model in 4-bit (Memory-Optimized)...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



🤖 Local Gemma Judge is processing the transcripts from your files...


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (8192). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clea


--- ⚖️ LLM Judge Report ---
**

** 

** | ** | 

** |

** |
**

** | ** 

** | 

** | 

** | 

``` |
**N**
** |

** |  | | 

** | ** |  | ** | ** | | 

** 

** | 

| | 

** | 


** | 

** | ** | ** | 

** | 

** | 
** 

** 
** | 
** | 
** | 

 

**
** |

** | 

** |


 

 | 
|
 |

** | 
 | 


 

** 

** |  

**
  |
**

** 

** |

** 

** | 

| 
 



| | | 

** 

** 
| | 

| 
 

** 
 

 
| |

** | 



** 

** | 



** 
 

| |


 ** | 

| 


** | 



** |  | 

** | | 

| 

| 

| 





📝 Successfully saved the evaluation report to: LLM_Judge_ASR_Evaluation.md
You can download it directly from your Files sidebar pane on the left!


In [ ]:
# import os
# import torch
# import gc
# from google.colab import userdata
# from transformers import pipeline, BitsAndBytesConfig

# # 1. Authenticate using your Hugging Face secret
# try:
#     os.environ["HF_TOKEN"] = userdata.get('HF_token')
#     print("✅ Hugging Face token loaded successfully!")
# except Exception as e:
#     print(f"❌ Failed to load secret: {e}")

# # 2. Check and map all available transcript files directly from your workspace sidebar
# files_to_check = {
#     "Garden2006/whisper-large-v3-turbo-gurmukhi-lora": "Garden2006_turbo.txt",
#     "abhi8799/whisper-large-v3-turbo-gurmukhi-lora": "abhi8799_turbo.txt",
#     "abhi8799/whisper-small-gurmukhi-lora": "abhi8799_small.txt",
#     "KaliNangia/whisper-large-v3-turbo-gurmukhi-lora": "KaliNangia_turbo.txt",
#     "KaliNangia/whisper-medium-gurmukhi-lora": "KaliNangia_medium.txt"
# }

# transcripts_block = ""
# print("\n📂 Scanning storage for ASR transcripts...")
# for model_name, filename in files_to_check.items():
#     if os.path.exists(filename):
#         with open(filename, "r", encoding="utf-8") as f:
#             # ✂️ Slice to 2500 characters to keep total tokens safely below context limits
#             snippet = f.read().strip()[:2500]
#             transcripts_block += f"### Model Name: {model_name}\nTranscript Snippet:\n{snippet}\n\n"
#             print(f"  ✔️ Successfully appended snippet from: {filename}")
# else:
#     print(f"  ❌ Missing file: {filename} (Skipped dynamically)")

# # 3. Setup global 4-bit config to protect memory capacity
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# # 4. Craft evaluation system prompt
# judge_prompt = f"""You are an expert linguistic judge specializing in Punjabi (Gurmukhi script) Automatic Speech Recognition (ASR).
# Your task is to comparatively evaluate the following transcript snippets based on linguistic fluency, Punjabi grammar, and presence of looping/hallucinations.

# {transcripts_block}

# Please compile your analysis strictly in the following Markdown table format followed by a concise summary declaring the clear winner:
# | Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
# | --- | --- | --- | --- |
# """
# messages = [{"role": "user", "content": judge_prompt}]

# # -----------------------------------------------------------------
# # 🤖 PIPELINE 1: Gemma 2 2B (E2B)
# # -----------------------------------------------------------------
# print("\n⏳ Loading local Gemma-2-2b-it (E2B)...")
# try:
#     pipeline_2b = pipeline(
#         "text-generation",
#         model="google/gemma-2-2b-it",
#         model_kwargs={"quantization_config": quantization_config}
#     )
#     print("🤖 Processing with Gemma 2B...")
#     with torch.no_grad():
#         report_2b = pipeline_2b(messages, max_new_tokens=800)[0]["generated_text"][-1]["content"]
#     del pipeline_2b
# except Exception as err:
#     report_2b = f"Execution failed: {err}"

# # Flush GPU VRAM cache safely before loading next target
# gc.collect()
# torch.cuda.empty_cache()

# # -----------------------------------------------------------------
# # 🤖 PIPELINE 2: Gemma 3 4B (E4B)
# # -----------------------------------------------------------------
# print("\n⏳ Loading local Gemma-3-4b-it (E4B)...")
# try:
#     pipeline_4b = pipeline(
#         "text-generation",
#         model="google/gemma-3-4b-it",
#         model_kwargs={"quantization_config": quantization_config}
#     )
#     print("🤖 Processing with Gemma 4B...")
#     with torch.no_grad():
#         report_4b = pipeline_4b(messages, max_new_tokens=800)[0]["generated_text"][-1]["content"]
#     del pipeline_4b
# except Exception as err:
#     report_4b = f"Execution failed: {err}"

# gc.collect()
# torch.cuda.empty_cache()

# # -----------------------------------------------------------------
# # 🤖 PIPELINE 3: Gemma 2 9B (12B Unified Class)
# # -----------------------------------------------------------------
# print("\n⏳ Loading local Gemma-2-9b-it (12B Tier)...")
# try:
#     pipeline_9b = pipeline(
#         "text-generation",
#         model="google/gemma-2-9b-it",
#         model_kwargs={"quantization_config": quantization_config}
#     )
#     print("🤖 Processing with Gemma 9B...")
#     with torch.no_grad():
#         report_9b = pipeline_9b(messages, max_new_tokens=800)[0]["generated_text"][-1]["content"]
#     del pipeline_9b
# except Exception as err:
#     report_9b = f"Execution failed: {err}"

# gc.collect()
# torch.cuda.empty_cache()

# # -----------------------------------------------------------------
# # 📊 GENERATED OUTPUT REPORTS DISPATCHER
# # -----------------------------------------------------------------
# print("\n" + "="*50 + "\n⚖️ MULTI-TIER LOCAL LLM JUDGE BENCHMARK REPORT\n" + "="*50)
# print("\n--- 📝 GEMMA 2B (E2B) ANALYSIS REPORT ---")
# print(report_2b)
# print("\n--- 📝 GEMMA 3 4B (E4B) ANALYSIS REPORT ---")
# print(report_4b)
# print("\n--- 📝 GEMMA 2 9B (12B ARCHITECTURE) REPORT ---")
# print(report_9b)

# # 6. Save combined transcripts matrix summary report physically
# output_file_path = "LLM_Judge_ASR_Evaluation.md"
# with open(output_file_path, "w", encoding="utf-8") as f:
#     f.write(f"# Gemma Benchmark Matrix Evaluations\n\n## 2B Tier\n{report_2b}\n\n## 4B Tier\n{report_4b}\n\n## 9B/12B Tier\n{report_9b}")
# print(f"\n💾 Compilation report updated inside local path workspace: {output_file_path}")

✅ Hugging Face token loaded successfully!

📂 Scanning storage for ASR transcripts...
  ✔️ Successfully appended snippet from: Garden2006_turbo.txt
  ✔️ Successfully appended snippet from: abhi8799_turbo.txt
  ✔️ Successfully appended snippet from: abhi8799_small.txt
  ✔️ Successfully appended snippet from: KaliNangia_turbo.txt
  ✔️ Successfully appended snippet from: KaliNangia_medium.txt
  ❌ Missing file: KaliNangia_medium.txt (Skipped dynamically)

⏳ Loading local Gemma-2-2b-it (E2B)...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🤖 Processing with Gemma 2B...


[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (8192). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean

...

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Processing with Gemma 9B...

⚖️ MULTI-TIER LOCAL LLM JUDGE BENCHMARK REPORT

--- 📝 GEMMA 2B (E2B) ANALYSIS REPORT ---
| KaliNangia/whisper-large-v3-turbo-gurmukhi-lora | 8 |  Strong performance, efficient, natural-sounding speech generation  | No | 
| 


--- 📝 GEMMA 3 4B (E4B) ANALYSIS REPORT ---
Okay, here's the analysis of the provided transcripts, followed by the table and summary.

**Analysis of Transcript Snippets**

Here's a breakdown of the transcript snippets, considering fluency, grammar, and the presence of looping/hallucinations, focusing on the `KaliNangia/whisper-medium-gurmukhi-lora` model:

**Transcript 1: `abhi8799/whisper-large-v3-turbo-gurmukhi-lora`**

*   **Score:** 6/10
*   **Fluency:** The fluency is noticeably lower than the other two. The sentence structure is quite convoluted and repetitive. There's a lot of unnecessary repetition of phrases ("ਸੰਪਰਚਨਾ", "ਮੈਂ", "ਕਿਸਾਨ").
*   **Grammar:** The grammar is present, but the use of words feels somewhat forced and un

In [5]:
import os
import torch
import gc
from google.colab import userdata
from transformers import pipeline, BitsAndBytesConfig

# Clear cache first
gc.collect()
torch.cuda.empty_cache()

# Authenticate using your exact lowercase secret key name
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_token')
    print("✅ Hugging Face token loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load secret: {e}")

# ... (rest of your file reading and quantization configuration setup)


files_to_check = {
    "Garden2006/whisper-large-v3-turbo-gurmukhi-lora": "Garden2006_turbo.txt",
    "abhi8799/whisper-large-v3-turbo-gurmukhi-lora": "abhi8799_turbo.txt",
    "abhi8799/whisper-small-gurmukhi-lora": "abhi8799_small.txt",
    "KaliNangia/whisper-large-v3-turbo-gurmukhi-lora": "KaliNangia_turbo.txt",
    "KaliNangia/whisper-medium-gurmukhi-lora": "KaliNangia_medium.txt"
}

transcripts_block = ""
for model_name, filename in files_to_check.items():
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            # Sliced tightly to 1200 chars to protect the 2B context window
            transcripts_block += f"### Model Name: {model_name}\nTranscript Snippet:\n{f.read().strip()[:1200]}\n\n"

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

judge_prompt = f"""You are an expert linguistic judge specializing in Punjabi ASR evaluation.
Comparatively evaluate these transcript snippets based on linguistic fluency, Punjabi grammar, and presence of loops:

{transcripts_block}

Output your analysis strictly in a Markdown table followed by a short summary of the winner:
| Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
| --- | --- | --- | --- |
"""

print("⏳ Loading local Gemma-2-2b-it...")
pipeline_2b = pipeline("text-generation", model="google/gemma-2-2b-it", model_kwargs={"quantization_config": quantization_config})

print("🤖 Processing Gemma 2B Report...")
with torch.no_grad():
    report_2b = pipeline_2b([{"role": "user", "content": judge_prompt}], max_new_tokens=800)[0]["generated_text"][-1]["content"]

print("\n--- ⚖️ GEMMA 2B REPORT ---")
print(report_2b)

with open("LLM_Judge_Gemma_2B_Evaluation.md", "w", encoding="utf-8") as f:
    f.write(report_2b)
print("💾 Saved Gemma 2B report to files sidebar!")

✅ Hugging Face token loaded successfully!
⏳ Loading local Gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Processing Gemma 2B Report...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- ⚖️ GEMMA 2B REPORT ---
## Model Analysis 

| Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
|---|-------------|---------------------------------|------------------------------------|
| Garden2006/whisper-large-v3-turbo-gurmukhi-lora | 8 |  Fluent, conveys information clearly,  | No |
| abhi8799/whisper-small-gurmukhi-lora | 7 |  Conveys information clearly, but less grammatically fluent than Garden2006 | No |
| KaliNangia/whisper-medium-gurmukhi-lora | 6 |  Conveys information clearly, but less fluent than Garden2006 | No | 

**Summary:**

**Garden2006/whisper-large-v3-turbo-gurmukhi-lora** appears to be the most fluent and grammatically robust model, with clear information delivery.  **abhi8799/whisper-small-gurmukhi-lora** is also good, but less fluent than Garden2006. **KaliNangia/whisper-medium-gurmukhi-lora** is a decent model with clear information delivery. 


**Note:** These scores are subjective and based on a limited anal

In [8]:
import torch
import gc
from transformers import pipeline, BitsAndBytesConfig

# Clear memory from previous run
gc.collect()
torch.cuda.empty_cache()

print("⏳ Loading local Gemma-3-4b-it...")
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
pipeline_4b = pipeline("text-generation", model="google/gemma-3-4b-it", model_kwargs={"quantization_config": quantization_config})

print("🤖 Processing Gemma 4B Report...")
with torch.no_grad():
    report_4b = pipeline_4b([{"role": "user", "content": judge_prompt}], max_new_tokens=1500)[0]["generated_text"][-1]["content"]

print("\n--- ⚖️ GEMMA 4B REPORT ---")
print(report_4b)

with open("LLM_Judge_Gemma_4B_Evaluation.md", "w", encoding="utf-8") as f:
    f.write(report_4b)
print("💾 Saved Gemma 4B report to files sidebar!")

⏳ Loading local Gemma-3-4b-it...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Processing Gemma 4B Report...

--- ⚖️ GEMMA 4B REPORT ---
Okay, let's analyze the provided Whisper transcript snippets.

**Analysis Table:**

| Model Name | Score (1-10) | Key Strength / Reason for Score | Hallucinations Detected (Yes/No) |
|---|---|---|---|
| Garden2006/whisper-large-v3-turbo-gurmukhi-lora | 6 | Relatively good fluency, good grammar mostly consistent, manages to maintain some context within the longer snippet.  Some minor repetition of phrases (e.g., "ਖਿੱਤੀ ਕਰਦੇ ਹਨ"). | No |
| abhi8799/whisper-large-v3-turbo-gurmukhi-lora | 7.5 | Very good fluency, near-perfect grammar.  Stronger contextual understanding, less repetition. Slightly better at maintaining flow. | No |
| abhi8799/whisper-small-gurmukhi-lora | 5 | Noticeable grammatical errors and inconsistencies.  Significant repetition (excessive use of phrases like "ਫੋਨ ਨੰਬਰ", "ਖਿੱਤੀ ਕਰਦੇ ਹਨ").  Less coherent overall. | No |
| KaliNangia/whisper-large-v3-turbo-gurmukhi-lora | 8.5 | Excellent fluency, very strong gramm

In [10]:
import torch
import gc
from transformers import pipeline, BitsAndBytesConfig

# Clear memory completely to prevent CUDA OOM
gc.collect()
torch.cuda.empty_cache()

print("⏳ Loading local Gemma-2-9b-it (12B Unified Architecture class)...")
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
pipeline_9b = pipeline("text-generation", model="google/gemma-2-9b-it", model_kwargs={"quantization_config": quantization_config})

print("🤖 Processing Gemma 9B Report...")
with torch.no_grad():
    report_9b = pipeline_9b([{"role": "user", "content": judge_prompt}], max_new_tokens=800)[0]["generated_text"][-1]["content"]

print("\n--- ⚖️ GEMMA 9B REPORT ---")
print(report_9b)

with open("LLM_Judge_Gemma_9B_Evaluation.md", "w", encoding="utf-8") as f:
    f.write(report_9b)
print("💾 Saved Gemma 9B report to files sidebar!")

⏳ Loading local Gemma-2-9b-it (12B Unified Architecture class)...


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Processing Gemma 9B Report...


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.15 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.12 GiB is free. Including non-PyTorch memory, this process has 13.45 GiB memory in use. Of the allocated memory 13.13 GiB is allocated by PyTorch, and 192.22 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)